# Анализ программы лояльности

## Задачи проекта:

1. провести исследовательский анализ данных показать общую картину
2. получить основные ритейл-метрики по когортам у клиентов внутри программы лояльности и вне ее
3. проанализировать насколько сработала текущая программа лояльности
4. если программа не слишком эффективна, то возможно, предложить способы повышения эффективности, обосновать использование других программ лояльности
5. если программа достаточно эффективна, то возможно, сказать каких еще клиентов стоит подключить к программе лояльности в первую очередь
6. Сформулировать и проверить гипотезы

## Описание таблиц

**retail_dataset**

- `purchaseId` — id чека
- `item_ID` — id товара
- `purchasedate` — дата покупки
- `Quantity` — количество товара
- `CustomerID` — id покупателя
- `ShopID` — id магазина
- `loyalty_program` — участвует ли покупатель в программе лояльности

**product_codes**
- `productID` — id товара
- `price_per_one` — стоимость одной единицы товара

## Загрузка библиотек и знакомство с данными

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [50]:
product_df = pd.read_csv('data/raw/product_codes.csv')
retail_df = pd.read_csv('data/raw/retail_dataset.csv')

In [51]:
def inform(df):
  print(f"___ Основная информация ___")
  print(f"Память: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
  display(df.head())
  print(f"___ Качество данных ___")
  profile = pd.DataFrame({
    "number_of_rows": df.shape[0],
    "number_of_cols": df.shape[1],
    "duplicates": df.duplicated().sum(),
    "dtype": df.dtypes.astype(str),
    "nunique": df.nunique(dropna=False),
    "gaps": df.isna().sum(),
    "gaps_share": (df.isna().mean() * 100).round(2)
  })
  display(profile.T)
  print(f"___ Описательная статистика ___")
  num = df.select_dtypes(include="number")
  if not num.empty:
    display(num.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))
    

In [52]:
# Посмотрим информацию про product_df
inform(product_df)

___ Основная информация ___
Память: 0.6 MB


,productID,price_per_one
0,85123A,2.55
1,71053,3.39
2,84406B,2.75
3,84029G,3.39
4,84029E,3.39


___ Качество данных ___


,productID,price_per_one
number_of_rows,9969,9969
number_of_cols,2,2
duplicates,0,0
dtype,str,float64
nunique,3159,586
gaps,0,0
gaps_share,0.0,0.0


___ Описательная статистика ___


,price_per_one
count,9969.00
mean,19.50
std,330.88
min,0.00
1%,0.00
5%,0.32
25%,1.25
50%,2.55
75%,5.51
95%,16.98


In [53]:
# Посмотрим информацию про retail_df
inform(retail_df)

___ Основная информация ___
Память: 27.0 MB


,purchaseid,item_ID,Quantity,purchasedate,CustomerID,ShopID,loyalty_program
0,538280,21873,11,2016-12-10 12:50:00,18427.0,Shop 0,0.0
1,538862,22195,0,2016-12-14 14:11:00,22389.0,Shop 0,1.0
2,538855,21239,7,2016-12-14 13:50:00,22182.0,Shop 0,1.0
3,543543,22271,0,2017-02-09 15:33:00,23522.0,Shop 0,1.0
4,543812,79321,0,2017-02-13 14:40:00,23151.0,Shop 0,1.0


___ Качество данных ___


,purchaseid,item_ID,Quantity,purchasedate,CustomerID,ShopID,loyalty_program
number_of_rows,105335,105335,105335,105335,105335,105335,105335
number_of_cols,7,7,7,7,7,7,7
duplicates,1033,1033,1033,1033,1033,1033,1033
dtype,str,str,int64,str,float64,str,float64
nunique,4894,3159,301,4430,1750,31,2
gaps,0,0,0,0,36210,0,0
gaps_share,0.0,0.0,0.0,0.0,34.38,0.0,0.0


___ Описательная статистика ___


,Quantity,CustomerID,loyalty_program
count,105335.00,69125.00,105335.00
mean,7.82,21019.30,0.23
std,327.95,1765.44,0.42
min,-74216.00,18025.00,0.00
1%,-3.00,18081.00,0.00
5%,0.00,18273.00,0.00
25%,0.00,19544.00,0.00
50%,2.00,20990.00,0.00
75%,7.00,22659.00,0.00
95%,24.00,23644.00,1.00


## Вывод
Таблица `product_df`:
- Датасет содержит 9969 строк и 2 колонки
- Дубликатов и пропусков нет
- Типы данных соответствуют содержимому признаков
- Минимальная цена равна `0`. Возможно, это товары с нулевой стоимостью или особенности формирования данных. На следующих этапах необходимо отдельно изучить такие значения.
- Максимальная цена составляет `16 888`, что значительно выше 99-го перцентиля `226.63`. Такие значения могут быть потенциальными выбросами, однако на данном этапе преждевременно считать их ошибочными так же необходимо дополнительно проверить их природу.
- Название колонок необходимо привести к единому виду

Таблица `retail_df`:
- Датасет содержит 105335 строк и 7 колонок
- Обнаруженно 1033 дубликатов и 36210 пропусков в колонке `CustomerID` необходимо определить причину этого
- Колонку `purchasedate` необходимо привести к типу `datetime`. А в столбцах `CustomerID` и `loyalty_program` следует поменять на  `int`.
- В столбце `Quantity` есть отрицательные значения, так же необходимо понять причину этого
- Так же название колонок необходимо привести к единому виду

## Предобработка данных

Приведем название колок к единому виду

In [54]:
product_df = product_df.rename(columns={'productID': 'product_id'})

In [55]:
product_df.columns

Index(['product_id', 'price_per_one'], dtype='str')

In [56]:
retail_df = retail_df.rename(columns={
                              'purchaseid': 'purchase_id',
                              'item_ID': 'item_id',
                              'Quantity': 'quantity',
                              'purchasedate': 'purchase_date',
                              'CustomerID': 'customer_id',
                              'ShopID': 'shop_id',
                            })

In [57]:
retail_df.columns

Index(['purchase_id', 'item_id', 'quantity', 'purchase_date', 'customer_id',
       'shop_id', 'loyalty_program'],
      dtype='str')

Проверим дубликаты в `retail_df`

In [58]:
retail_df[retail_df.duplicated(keep=False)].sort_values(by=retail_df.columns.tolist()).head(6)

,purchase_id,item_id,quantity,purchase_date,customer_id,shop_id,loyalty_program
35797,536409,21866,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0
56087,536409,21866,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0
22208,536409,22111,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0
54076,536409,22111,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0
24235,536409,22866,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0
60212,536409,22866,0,2016-12-01 11:45:00,23587.0,Shop 0,1.0


In [59]:
duplicated_share = (retail_df.duplicated().sum() / len(retail_df) * 100).round(2)
print(f'Процент дублей от общего кол-ва зписей: {duplicated_share}%')

Процент дублей от общего кол-ва зписей: 0.98%


Дублирующиеся строки составляют менее `1%` от общего объема данных и, вероятно являются результатом технического сбоя или ошибочной записи, поэтому они могут быть безопасно удалены

In [60]:
retail_df = retail_df.drop_duplicates()
retail_df.duplicated().sum()

np.int64(0)

Обработка пропусков

Проверим если чек `purchase_id` содержит несколько позиций, `customer_id` может быть заполнен только в одной строке. В этом случае пропуски можно восполнить по `purchase_id` внутри чека. 

In [61]:
retail_dropna = retail_df.dropna().reset_index()       
retail_na = retail_df[retail_df['customer_id'].isnull()].reset_index() 
retail_na_new = retail_na.merge(retail_dropna, on='purchase_id', how='inner')
len(retail_na_new)

0

Пересечений по `purchase_id` между строками с пропуском и полными строками нет. Восстановить `customer_id` по номеру чека невозможно.

Если все клиенты в датасете покупали ровно один раз, каждой строке с пропуском можно было бы присвоить свой уникальный id. Проверим частоту повторных визитов среди идентифицированных клиентов: если повторные покупки распространены, анонимный клиент мог оставить несколько чеков, и присвоение уникального id каждой строке исказит метрики.

In [62]:
retail_dropna.groupby('customer_id')['purchase_id'].nunique().reset_index()['purchase_id'].describe()

count    1749.000000
mean        2.370497
std         2.976829
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        46.000000
Name: purchase_id, dtype: float64

Повторные покупки характерны минимум для четверти клиентов, у анонимных покупателей с тем же поведением могло быть несколько чеков. Мы не можем сопоставить их между собой, а присвоение уникального id каждому чеку искусственно раздует клиентскую базу.

In [63]:
missing_share = retail_df['customer_id'].isna().mean()
print(f'Доля пропусков в customer_id от общего числа покупок: {missing_share:.1%}')

Доля пропусков в customer_id от общего числа покупок: 34.7%


Доля пропусков в `сustomer_id` `34.7%`. Удалять иx нецелесообразно. Так как восстановить такие пропуски не представляется возможным,заполним эти пропуски нулями.

In [64]:
retail_df['customer_id'] = retail_df['customer_id'].fillna(0)
retail_df.isna().sum()

purchase_id        0
item_id            0
quantity           0
purchase_date      0
customer_id        0
shop_id            0
loyalty_program    0
dtype: int64

Приведем колонки `customer_id` и `loyalty_program` к типу данных `int`. А колонку `purchase_date` к `datetime`

In [65]:
retail_df[['customer_id', 'loyalty_program']] = retail_df[['customer_id', 'loyalty_program']].astype(int)

In [66]:
retail_df['purchase_date'] = pd.to_datetime(retail_df['purchase_date'], format='%Y-%m-%d %H:%M:%S')

In [67]:
retail_df.dtypes

purchase_id                   str
item_id                       str
quantity                    int64
purchase_date      datetime64[us]
customer_id                 int64
shop_id                       str
loyalty_program             int64
dtype: object

In [69]:
product_df['product_id'].nunique()

3159

Вероятно некоторые товары имеют несколько вариантов цен. Это может быть связано с изменением цены во времени или разными ценами в магазинах сети, что для ритейла вполне нормально. Для анализа нам нужна одна цена на товар, поэтому сначала оценим масштаб проблемы. 

In [70]:
prices_count = product_df.groupby('product_id')['price_per_one'].nunique().reset_index()
prices_count[prices_count['price_per_one'] > 1].shape[0]

2494

Почти `2500` товаров имеют несколько цен. В качестве единой цены возьмём медиану она устойчива к выбросам и не завышает-незанижает метрики.

In [73]:
product_df = product_df.pivot_table(index='product_id', values='price_per_one', aggfunc='median')
product_df

,price_per_one
product_id,
10002,1.630
10080,0.850
10120,0.210
10123C,0.650
10124A,0.420
...,...
gift_0001_20,16.845
gift_0001_30,25.265
gift_0001_40,34.040


Добавим цены на товары к основному датасету и рассчитаем выручку по позиции

In [74]:
retail = retail_df.merge(product_df, how='left', left_on='item_id', right_on='product_id')

In [76]:
retail['revenue'] = retail['quantity'] * retail['price_per_one']

In [77]:
retail.head()

,purchase_id,item_id,quantity,purchase_date,customer_id,shop_id,loyalty_program,price_per_one,revenue
0,538280,21873,11,2016-12-10 12:50:00,18427,Shop 0,0,1.63,17.93
1,538862,22195,0,2016-12-14 14:11:00,22389,Shop 0,1,3.29,0.00
2,538855,21239,7,2016-12-14 13:50:00,22182,Shop 0,1,1.63,11.41
3,543543,22271,0,2017-02-09 15:33:00,23522,Shop 0,1,4.37,0.00
4,543812,79321,0,2017-02-13 14:40:00,23151,Shop 0,1,5.75,0.00


### Итоги
1. Названия столбцов приведены к единому виду
2. Удалено `1033` дубликатов. Их доля составляла `0.98%`
3. Пропуски в `сustomer_id` составляли `34.7%` от общего числа строк. Выяснили что удалять их нецелесообразно, поэтому заменили их на `0`
4. Столбцы `loyalty_program` и `customer_id` привели к типу данных `int`, а `purchase_date` к типу `datetime`
5. Определили что один товар мог иметь несколько цен, что для ритейла нормально. Но для дальнейшего анализа заменили цену продукта на медианное значение
6. Объединены таблицы `retail_df` и `product_df`. Рассчитал и добавил  новую колонку `revenue`